In [1]:
import os
import torch
import random
import numpy as np
from torch import nn
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from torch.optim import Adam
import matplotlib.pyplot as plt
from torchvision.transforms import v2
import segmentation_models_pytorch as smp
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Dataset, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def build_transforms(crop_size=256, augment=True):
    # paired transforms (image & mask transformed together)
    if augment:
        return v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),

            v2.RandomCrop(crop_size),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.2)
        ])
    else:
        return v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True)
        ])

class CrackSegmentationDataset(Dataset):
    def __init__(self, paths, crop_size=256, augment=True):
        self.paths = paths

        self.images = []
        self.masks = []

        for img_path, mask_path in paths:
            self.images.append(Image.open(img_path).convert("RGB"))
            self.masks.append(Image.open(mask_path).convert("1"))
        
        self.transform = build_transforms(crop_size, augment)
        self.crop_size = crop_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image, mask = self.transform(self.images[idx], self.masks[idx])
        return image, mask